## 15. Unsupervised Anomaly Detection:
We are combining two distinct algorithms to form a resilient, vote-based evaluation.
1. **Feature Standardization (`StandardScaler`):**
Both of our models rely on geometric distance metrics. If we don't scale the data, the model will prioritize features with large raw numbers (like Price = $150$) over vital percentages (like Spread = $0.02$). `StandardScaler` forces every feature to have a mean of 0 and a standard deviation of 1, leveling the playing field.
2. **Isolation Forest (Global Outliers):**
It randomly slices features to build decision trees. Anomalies are points that get isolated extremely fast (short path lengths) because they are far away from the dense cluster of normal math.
    *  **Parameters**: `n_estimators=200` (We build 200 trees to ensure a stable, robust average score). `contamination=0.01` (We strictly tell the model that only the extreme 1% of the 8-year history are true anomalies).

3. **Local Outlier Factor / LOF (Local Outliers):**
Instead of looking at the global 8-year picture, LOF looks at a data point's immediate neighbors in hyperspace. If a point is sitting in a low-density void while its neighbors are densely packed, it's an anomaly.

    *  **Parameters**: `n_neighbors=20` (We instruct the algorithm to compare each hour to its 20 mathematically closest peers).

4. **The Ensemble Vote:**
We normalize the scores from both models to a strict 0-to-1 scale. We then take the average of both scores. We finalize an `anomaly_flag` ONLY if the average score falls in the top 1% across the entire dataset. This prevents one hyper-sensitive model from producing False Positives.

In [1]:
# Notebook imports
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

In [2]:
# Notebook configuration
FEATURES = [
    "spread_pct",
    "vwap_deviation",
    "return_1h",
    "log_return",
    "high_low_range",
    "vol_6h",
    "vol_24h",
]
WINDOW_Z = 30
ROLL_WINDOW = 20
CONTAMINATION = 0.01
N_ESTIMATORS = 200
N_NEIGHBORS = 20

COLORS = {
    "price": "#2563eb",
    "anomaly": "#dc2626",
    "normal": "#6b7280",
}

In [3]:
_cwd = Path().resolve()
PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
DATA_DIR = PROJECT_ROOT / "data"
PLOTS_DIR = PROJECT_ROOT / "reports" / "figures"

# Self-load
PROCESSED_CSV = DATA_DIR / "processed" / "msft_hourly(in)_processed.csv"
if not PROCESSED_CSV.exists():
    raise FileNotFoundError(
        f"Processed CSV not found at {PROCESSED_CSV}. "
        "Run 1.0-abz-preprocessing.ipynb first to generate it."
    )

df = pd.read_csv(PROCESSED_CSV, parse_dates=["quote_datetime"])


def compute_features(data):
    """Compute all engineered features needed for modeling."""
    features = data.copy()
    features["return_1h"] = features["close"].pct_change()
    features["log_return"] = np.log(features["close"] / features["close"].shift(1))
    features["vol_6h"] = features["log_return"].rolling(6).std()
    features["vol_24h"] = features["log_return"].rolling(24).std()
    features["spread_pct"] = (features["ask"] - features["bid"]) / features["mid"] * 100
    features["vwap_deviation"] = (
        (features["close"] - features["vwap"]) / features["vwap"] * 100
    )
    features["high_low_range"] = (
        (features["high"] - features["low"]) / features["close"] * 100
    )
    return features


FEATURE_COLS = FEATURES  # align with config above
feature_cols = FEATURES  # lowercase alias for PCA cell

df_feat = compute_features(df)
df_model = df_feat.dropna(subset=FEATURES).copy().reset_index(drop=True)
print(f"Self-load complete: {len(df_model)} rows, {len(FEATURES)} features")

Self-load complete: 15477 rows, 7 features


In [4]:
# Isolation Forest (IF)

if "df_model" not in globals():
    raise ValueError("df_model not found. Run feature-engineering cell first.")

features = FEATURES
part = df_model.copy().sort_values("quote_datetime").reset_index(drop=True)
active_features = [c for c in features if c in part.columns]
window_z = WINDOW_Z
roll_window = ROLL_WINDOW

for col in active_features:
    mu = part[col].rolling(window_z).mean()
    sd = part[col].rolling(window_z).std()
    part[f"{col}_z"] = (part[col] - mu) / (sd + 1e-8)

ml_features = []
for col in active_features:
    zc = f"{col}_z"
    if zc in part.columns:
        ml_features.extend([col, zc])

part = part.dropna(subset=ml_features).reset_index(drop=True)
X = part[ml_features].values
X_scaled = StandardScaler().fit_transform(X)

iforest = IsolationForest(
    n_estimators=N_ESTIMATORS,
    contamination=CONTAMINATION,
    random_state=42,
    n_jobs=-1,
)
iforest.fit(X_scaled)

part["iforest_flag"] = (iforest.predict(X_scaled) == -1).astype(int)
part["iforest_score"] = -iforest.decision_function(X_scaled)

mu2 = part["iforest_score"].rolling(roll_window).mean()
sd2 = part["iforest_score"].rolling(roll_window).std()
part["if_z20"] = (part["iforest_score"] - mu2) / (sd2 + 1e-8)

flagged = part[part["iforest_flag"] == 1]

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=("Price with Outliers", "IF Score"),
)

fig.add_trace(
    go.Scatter(
        x=part["quote_datetime"],
        y=part["close"],
        name="Close Price",
        line=dict(color=COLORS["normal"], width=1),
        opacity=0.5,
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=flagged["quote_datetime"],
        y=flagged["close"],
        mode="markers",
        name="Outliers",
        marker=dict(color=COLORS["anomaly"], size=6),
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=part["quote_datetime"],
        y=part["if_z20"],
        name="IF Score",
        line=dict(color=COLORS["price"], width=0.8),
    ),
    row=2,
    col=1,
)

fig.add_hline(y=3, line_dash="dash", line_color="red", row=2, col=1)
fig.add_hline(y=-3, line_dash="dash", line_color="red", row=2, col=1)

fig.update_layout(
    height=800,
    title_text="Isolation Forest Outlier Detection Dashboard",
    template="plotly_white",
    showlegend=True,
)
fig.show()

print("Isolation Forest anomalies:", int(part["iforest_flag"].sum()))

Isolation Forest anomalies: 155


In [5]:
# Local Outlier Factor (LOF)

if "df_model" not in globals():
    raise ValueError("df_model not found. Run feature-engineering cell first.")

features = FEATURES
part = df_model.copy().sort_values("quote_datetime").reset_index(drop=True)
active_features = [c for c in features if c in part.columns]
window_z = WINDOW_Z
roll_window = ROLL_WINDOW

for col in active_features:
    mu = part[col].rolling(window_z).mean()
    sd = part[col].rolling(window_z).std()
    part[f"{col}_z"] = (part[col] - mu) / (sd + 1e-8)

ml_features = []
for col in active_features:
    zc = f"{col}_z"
    if zc in part.columns:
        ml_features.extend([col, zc])

part = part.dropna(subset=ml_features).reset_index(drop=True)
X = part[ml_features].values
X_scaled = StandardScaler().fit_transform(X)

lof = LocalOutlierFactor(
    n_neighbors=N_NEIGHBORS, contamination=CONTAMINATION, n_jobs=-1
)
part["lof_flag"] = (lof.fit_predict(X_scaled) == -1).astype(int)
part["lof_score"] = -lof.negative_outlier_factor_

mu2 = part["lof_score"].rolling(roll_window).mean()
sd2 = part["lof_score"].rolling(roll_window).std()
part["lof_z20"] = (part["lof_score"] - mu2) / (sd2 + 1e-8)

flagged = part[part["lof_flag"] == 1]

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=("Price with Outliers", "LOF Score"),
)

fig.add_trace(
    go.Scatter(
        x=part["quote_datetime"],
        y=part["close"],
        name="Close Price",
        line=dict(color=COLORS["normal"], width=1),
        opacity=0.5,
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=flagged["quote_datetime"],
        y=flagged["close"],
        mode="markers",
        name="Outliers",
        marker=dict(color=COLORS["anomaly"], size=6),
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=part["quote_datetime"],
        y=part["lof_z20"],
        name="LOF Score",
        line=dict(color=COLORS["price"], width=0.8),
    ),
    row=2,
    col=1,
)

fig.add_hline(y=3, line_dash="dash", line_color="red", row=2, col=1)
fig.add_hline(y=-3, line_dash="dash", line_color="red", row=2, col=1)

fig.update_layout(
    height=800,
    title_text="LOF Outlier Detection Dashboard",
    template="plotly_white",
    showlegend=True,
)
fig.show()

print("LOF anomalies:", int(part["lof_flag"].sum()))

LOF anomalies: 155
